In [1]:
# Import required libraries
import sys
import time
import pandas as pd

# Allow notebook to import files from src/
sys.path.append("../src")

from llm_client import generate_response
from prompts import (
    analysis_prompt_v1,
    analysis_prompt_v2,
    summarization_prompt_v1,
    summarization_prompt_v2,
    classification_prompt_v1,
    classification_prompt_v2,
    classification_prompt_v3,
    generation_prompt_v1,
    generation_prompt_v2
)

In [3]:
# Load the balanced prompt experiment dataset
df = pd.read_csv(
    "../data/processed/prompt_experiment_tickets.csv"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (120, 8)


,ticket_id,subject,body,answer,type,queue,priority,language
0,1,Problem with Server During Peak Hours,"To Whom It May Concern, I hope this message fi...","Dear [Customer Name], Thank you for your email...",Incident,Service Outages and Maintenance,medium,en
1,2,Concern Regarding Data Security Breach,Data security breaches have affected the hospi...,Immediate investigation into the data security...,Incident,Technical Support,low,en
2,3,Modify Employee Training Curriculum for Data A...,I am contacting you to suggest updating the em...,"Dear <name>, we thank you for proposing to upd...",Change,Human Resources,medium,en
3,4,Project Management Portal Freezing Issues,The project management portal has frozen unexp...,I will look into the project management portal...,Incident,Customer Service,medium,en
4,5,No subject,"Dear Customer Support, I am writing to request...",Thank you for reaching out to us to enhance yo...,Change,Technical Support,high,en


In [6]:
# Select a few tickets for initial prompt testing
test_df = df.head(3).copy()

test_df[
    ["ticket_id", "subject", "type", "priority"]
]

,ticket_id,subject,type,priority
0,1,Problem with Server During Peak Hours,Incident,medium
1,2,Concern Regarding Data Security Breach,Incident,low
2,3,Modify Employee Training Curriculum for Data A...,Change,medium


In [4]:
# Test baseline summarization prompt
row = test_df.iloc[0]

prompt = summarization_prompt_v1(
    row["subject"],
    row["body"]
)

summary_v1 = generate_response(prompt)

print("ORIGINAL TICKET:\n")
print(row["body"])

print("\n--- V1 SUMMARY ---\n")
print(summary_v1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ORIGINAL TICKET:

To Whom It May Concern, I hope this message finds you well. I am reaching out due to a server outage that happened during our busiest period. It might have been caused by a server overload. My team has tried to address the issue by restarting the servers and reviewing the logs, but the problem still exists. I kindly request your assistance in looking into this issue promptly and offering a solution to prevent similar occurrences in the future. The downtime has greatly affected our operations, and we urgently need to return to normal. Please let us know if there is any information you require from us to help investigate and resolve this matter. Thank you for your swift response. We are looking forward to hearing from you soon.

--- V1 SUMMARY ---

The customer is experiencing a server outage that occurred during peak hours, possibly due to a server overload. Their team has already attempted to resolve the issue by restarting servers and reviewing logs, but the problem 

In [5]:
# Test baseline classification prompt
row = test_df.iloc[0]

prompt = classification_prompt_v1(
    row["subject"],
    row["body"]
)

classification_v1 = generate_response(prompt)

print("ACTUAL LABEL:")
print(row["type"])

print("\nMODEL OUTPUT:")
print(classification_v1)

ACTUAL LABEL:
Incident

MODEL OUTPUT:
This ticket should be classified as an **Incident**.

Here's why:

*   **Incident:** An unplanned interruption to an IT service or a reduction in the quality of an IT service. The primary goal of incident management is to restore normal service operation as quickly as possible and minimize the adverse impact on business operations.

The ticket describes:
*   "a server outage that happened during our busiest period."
*   "the problem still exists."
*   "The downtime has greatly affected our operations, and we urgently need to return to normal."
*   A request for assistance in "looking into this issue promptly and offering a solution."

While the ticket also asks for a solution "to prevent similar occurrences in the future" (which is related to Problem Management), the immediate and urgent need is to resolve an active or very recent service disruption that is impacting operations. Therefore, it is first and foremost an **Incident**.


In [6]:
# Test baseline response generation prompt
row = test_df.iloc[0]

prompt = generation_prompt_v1(
    row["subject"],
    row["body"]
)

generated_response_v1 = generate_response(prompt)

print("CUSTOMER TICKET:\n")
print(row["body"])

print("\n--- GENERATED RESPONSE ---\n")
print(generated_response_v1)

print("\n--- DATASET REFERENCE ANSWER ---\n")
print(row["answer"])

CUSTOMER TICKET:

To Whom It May Concern, I hope this message finds you well. I am reaching out due to a server outage that happened during our busiest period. It might have been caused by a server overload. My team has tried to address the issue by restarting the servers and reviewing the logs, but the problem still exists. I kindly request your assistance in looking into this issue promptly and offering a solution to prevent similar occurrences in the future. The downtime has greatly affected our operations, and we urgently need to return to normal. Please let us know if there is any information you require from us to help investigate and resolve this matter. Thank you for your swift response. We are looking forward to hearing from you soon.

--- GENERATED RESPONSE ---

Subject: Re: Problem with Server During Peak Hours - Ticket ID: [Automatically Generated Ticket ID]

Dear [Customer Name, or "Valued Customer" if specific name isn't known],

Thank you for reaching out and bringing th

In [7]:
# Prepare a small batch of tickets for data analysis
analysis_sample = df.head(10)[
    ["subject", "body", "type", "priority"]
]

ticket_data = analysis_sample.to_string(index=False)

prompt = analysis_prompt_v1(ticket_data)

analysis_v1 = generate_response(prompt)

print(analysis_v1)

Based on the provided customer support ticket data, here are the main patterns observed:

1.  **Dominant Themes: Security, Data & Analytics, and Operational Stability**
    *   **Security:** This is the most prevalent theme, appearing in 5 out of 10 tickets. It encompasses data security breaches, medical information leaks, unauthorized access attempts, and proactive security enhancements (encryption).
    *   **Data & Analytics:** There's a significant focus on leveraging data for strategic purposes, particularly for investment optimization and improving digital campaigns. This includes requests for data analytics products/services and training curriculum modifications.
    *   **Operational Stability:** Issues like server outages and application freezing highlight concerns about system reliability and performance.

2.  **Priority Correlates with Impact and Urgency, with Nuances in Security**
    *   **High Priority:** Assigned to confirmed critical issues (e.g., "Unanticipated Medical

In [8]:
# Test classification V1 on three tickets
classification_results = []

for _, row in test_df.iterrows():

    prompt = classification_prompt_v1(
        row["subject"],
        row["body"]
    )

    prediction = generate_response(prompt)

    classification_results.append({
        "ticket_id": row["ticket_id"],
        "actual_type": row["type"],
        "v1_output": prediction
    })

    # Small delay between API calls
    time.sleep(1)

classification_results_df = pd.DataFrame(
    classification_results
)

classification_results_df

Gemini server busy. Retrying in 5 seconds...


,ticket_id,actual_type,v1_output
0,1,Incident,This ticket should be classified as an **Incid...
1,2,Incident,This ticket should be classified as an **Incid...
2,3,Change,This ticket is a **Change**.\n\nHere's why:\n\...


## Baseline Prompt V1 Observations

Initial testing showed that the baseline prompts were functional but lacked
specific instructions and output constraints.

### Classification V1
- Correctly classified the initial test tickets.
- Returned additional explanations instead of only the class label.
- Output format was inconsistent across tickets.
- Did not define the meaning of each ticket category.

### Summarization V1
- Captured the main issue.
- No restriction on summary length.
- No explicit requirement to preserve key details.
- No instruction to avoid unsupported information.

### Content Generation V1
- No defined tone or maximum length.
- No restriction against unsupported promises or assumptions.
- Response structure was not specified.

### Data Analysis V1
- Request was broad and unstructured.
- Did not specify which patterns or metrics should be analyzed.
- Did not define a required output structure.

In [9]:
# Compare classification V1 and V2 on the same three tickets
comparison_results = []

for _, row in test_df.iterrows():

    prompt_v1 = classification_prompt_v1(
        row["subject"],
        row["body"]
    )

    prompt_v2 = classification_prompt_v2(
        row["subject"],
        row["body"]
    )

    output_v1 = generate_response(prompt_v1)
    time.sleep(1)

    output_v2 = generate_response(prompt_v2)
    time.sleep(1)

    comparison_results.append({
        "ticket_id": row["ticket_id"],
        "actual_type": row["type"],
        "v1_output": output_v1,
        "v2_output": output_v2
    })

classification_comparison_df = pd.DataFrame(
    comparison_results
)

classification_comparison_df

Gemini server busy. Retrying in 5 seconds...
Gemini server busy. Retrying in 10 seconds...


,ticket_id,actual_type,v1_output,v2_output
0,1,Incident,This ticket should be classified as an **Incid...,Incident
1,2,Incident,This ticket should be classified as an **Incid...,Problem
2,3,Change,This ticket clearly falls under the **Change**...,Change


In [7]:
# Save V1 vs V2 classification comparison
classification_comparison_df.to_csv(
    "../outputs/classification_v1_v2_comparison.csv",
    index=False
)

print("Comparison results saved.")

NameError: name 'classification_comparison_df' is not defined

In [11]:
# Clean V2 output
classification_comparison_df["v2_clean"] = (
    classification_comparison_df["v2_output"]
    .str.strip()
)

# Check whether prediction matches the actual label
classification_comparison_df["v2_correct"] = (
    classification_comparison_df["actual_type"]
    == classification_comparison_df["v2_clean"]
)

v2_accuracy = classification_comparison_df["v2_correct"].mean()

print(f"V2 Accuracy: {v2_accuracy:.2%}")

V2 Accuracy: 66.67%


In [5]:
# Test V3 only on the ticket that V2 misclassified

row = test_df.iloc[1]

prompt_v3 = classification_prompt_v3(
    row["subject"],
    row["body"]
)

output_v3 = generate_response(prompt_v3)

print("ACTUAL LABEL:")
print(row["type"])

print("\nV2 OUTPUT:")
print(
    classification_comparison_df.loc[
        classification_comparison_df["ticket_id"] == row["ticket_id"],
        "v2_output"
    ].iloc[0]
)

print("\nV3 OUTPUT:")
print(output_v3)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ACTUAL LABEL:
Incident

V2 OUTPUT:


NameError: name 'classification_comparison_df' is not defined

In [8]:
# Recreate the V1 vs V2 comparison results from the previous experiment

classification_comparison_df = pd.DataFrame({
    "ticket_id": [1, 2, 3],
    "actual_type": ["Incident", "Incident", "Change"],
    "v1_prediction": ["Incident", "Incident", "Change"],
    "v2_output": ["Incident", "Problem", "Change"]
})

classification_comparison_df

,ticket_id,actual_type,v1_prediction,v2_output
0,1,Incident,Incident,Incident
1,2,Incident,Incident,Problem
2,3,Change,Change,Change


In [9]:
# Save V1 vs V2 classification comparison
classification_comparison_df.to_csv(
    "../outputs/classification_v1_v2_comparison.csv",
    index=False
)

print("Comparison results saved.")

Comparison results saved.


In [10]:
pd.read_csv("../outputs/classification_v1_v2_comparison.csv")

,ticket_id,actual_type,v1_prediction,v2_output
0,1,Incident,Incident,Incident
1,2,Incident,Incident,Problem
2,3,Change,Change,Change


In [11]:
# Test V3 only on the ticket that V2 misclassified

row = test_df.iloc[1]

prompt_v3 = classification_prompt_v3(
    row["subject"],
    row["body"]
)

output_v3 = generate_response(prompt_v3)

print("ACTUAL LABEL:")
print(row["type"])

print("\nV2 OUTPUT:")
print(
    classification_comparison_df.loc[
        classification_comparison_df["ticket_id"] == row["ticket_id"],
        "v2_output"
    ].iloc[0]
)

print("\nV3 OUTPUT:")
print(output_v3)

ACTUAL LABEL:
Incident

V2 OUTPUT:
Problem

V3 OUTPUT:
Problem


In [12]:
# Inspect the ticket that V2 and V3 misclassified

row = test_df.iloc[1]

print("SUBJECT:")
print(row["subject"])

print("\nTICKET BODY:")
print(row["body"])

print("\nDATASET LABEL:")
print(row["type"])

print("\nV2 PREDICTION:")
print("Problem")

print("\nV3 PREDICTION:")
print(output_v3)

SUBJECT:
Concern Regarding Data Security Breach

TICKET BODY:
Data security breaches have affected the hospital's systems due to the use of outdated software.

DATASET LABEL:
Incident

V2 PREDICTION:
Problem

V3 PREDICTION:
Problem


### Classification V3 Observation

V3 retained the prediction `Problem` for ticket 2, while the dataset label was
`Incident`.

The ticket described data security breaches affecting hospital systems and
explicitly mentioned outdated software as the cause. Based on the prompt
definitions, the model interpreted this as an underlying root-cause issue,
which is consistent with the `Problem` category.

This indicates that some dataset labels may be ambiguous relative to the
prompt definitions. Therefore, a disagreement between the model and the
dataset does not necessarily indicate poor prompt quality.

V3 improved the clarity of the classification decision rules, but larger-scale
evaluation is required before drawing conclusions about classification accuracy.

In [13]:
# Create a small balanced evaluation sample
evaluation_df = (
    df
    .groupby("type", group_keys=False)
    .sample(n=3, random_state=100)
    .reset_index(drop=True)
)

evaluation_df["eval_id"] = range(1, len(evaluation_df) + 1)

print("Evaluation sample shape:", evaluation_df.shape)
print("\nClass distribution:")
print(evaluation_df["type"].value_counts())

evaluation_df[
    ["eval_id", "subject", "type"]
]

Evaluation sample shape: (12, 9)

Class distribution:
type
Change      3
Incident    3
Problem     3
Request     3
Name: count, dtype: int64


,eval_id,subject,type
0,1,No subject,Change
1,2,Support Request for Billing Details Update,Change
2,3,Enhancements to User Interface for Project Man...,Change
3,4,Problem with SaaS Dashboard Display,Incident
4,5,Urgent Assistance Required for Medical Data Se...,Incident
5,6,No subject,Incident
6,7,Challenges in Marketing Campaigns,Problem
7,8,Problem Regarding Data Encryption,Problem
8,9,Network Connectivity Problem,Problem
9,10,Support for Integrating Drupal with Magento,Request


In [14]:
# Convert evaluation tickets into a single batch for Gemini

def build_ticket_batch(data):
    tickets = []

    for _, row in data.iterrows():
        tickets.append(
            f"""
Ticket ID: {row['eval_id']}
Subject: {row['subject']}
Body: {row['body']}
"""
        )

    return "\n".join(tickets)


ticket_batch = build_ticket_batch(evaluation_df)

print(ticket_batch[:2000])


Ticket ID: 1
Subject: No subject
Body: Customer Support, we are reaching out to request an enhancement to improve the compatibility of our applications across multiple platforms, which would greatly enhance the user experience in project management. Currently, our team is facing challenges in seamlessly integrating different tools and software, which hinders our productivity and efficiency. Enhancing compatibility would allow us to work more smoothly and effectively. We believe that this improvement would greatly benefit our workflow and overall project outcomes. We would appreciate it if you could consider our request.


Ticket ID: 2
Subject: Support Request for Billing Details Update
Body: I am writing to request an update to my billing details for my digital marketing subscriptions. Currently, I have multiple subscriptions with different payment due dates, which is confusing and inefficient. I would like to streamline the payment process to improve efficiency. Could you please assi

In [15]:
def batch_classification_v1(ticket_batch):
    return f"""
Classify each customer support ticket into one of these types:

Incident
Request
Problem
Change

Tickets:
{ticket_batch}

Return exactly one line for every ticket using this format:

Ticket ID | Label

Example:
1 | Incident
2 | Request

Do not skip any ticket.
"""


def batch_classification_v2(ticket_batch):
    return f"""
You are a customer support ticket classification system.

Classify each ticket into exactly ONE category:

Incident - An unplanned interruption, failure, or reduction in service.

Request - A request for information, access, assistance,
or a standard service.

Problem - An underlying, recurring, or root-cause issue
requiring investigation.

Change - A request to modify, update, configure,
or change a system or service.

Tickets:
{ticket_batch}

Return exactly one line for every ticket using this format:

Ticket ID | Label

Example:
1 | Incident
2 | Request

Use only these labels:
Incident
Request
Problem
Change

Do not include explanations.
"""


def batch_classification_v3(ticket_batch):
    return f"""
You are a customer support ticket classification system.

Classify each ticket into exactly ONE category.

Definitions:

Incident:
A specific unplanned interruption, outage, failure,
degradation, or malfunction.

Request:
A request for information, access, assistance,
or a standard service.

Problem:
A recurring issue or underlying root-cause issue
requiring investigation.

Change:
A request to modify, update, configure, replace,
or change an existing system or service.

Decision rules:

- A current or one-time failure should normally be Incident.
- Use Problem for recurring issues or root-cause investigation.
- Do not classify a single failure as Problem only because
  a possible cause is mentioned.
- Use Change when modification is explicitly requested.
- Use Request for information, access, assistance,
  or normal service requests.

Tickets:
{ticket_batch}

Return exactly one line for every ticket using this format:

Ticket ID | Label

Example:
1 | Incident
2 | Request

Use only:
Incident
Request
Problem
Change

Do not include explanations.
"""

In [16]:
print("Running V1...")
batch_v1_output = generate_response(
    batch_classification_v1(ticket_batch)
)

time.sleep(5)

print("Running V2...")
batch_v2_output = generate_response(
    batch_classification_v2(ticket_batch)
)

time.sleep(5)

print("Running V3...")
batch_v3_output = generate_response(
    batch_classification_v3(ticket_batch)
)

print("\nCompleted.")

Running V1...
Running V2...
Running V3...

Completed.


In [17]:
print("=== V1 ===")
print(batch_v1_output)

print("\n=== V2 ===")
print(batch_v2_output)

print("\n=== V3 ===")
print(batch_v3_output)

=== V1 ===
1 | Change
2 | Request
3 | Change
4 | Incident
5 | Incident
6 | Incident
7 | Problem
8 | Incident
9 | Incident
10 | Request
11 | Request
12 | Request

=== V2 ===
1 | Change
2 | Request
3 | Change
4 | Problem
5 | Incident
6 | Incident
7 | Problem
8 | Problem
9 | Problem
10 | Request
11 | Request
12 | Request

=== V3 ===
1 | Change
2 | Request
3 | Change
4 | Problem
5 | Incident
6 | Incident
7 | Problem
8 | Problem
9 | Incident
10 | Request
11 | Request
12 | Request


In [18]:
import re


def parse_batch_predictions(output):
    predictions = {}

    valid_labels = {
        "incident",
        "request",
        "problem",
        "change"
    }

    for line in output.splitlines():

        match = re.search(
            r"(\d+)\s*\|\s*(Incident|Request|Problem|Change)",
            line,
            re.IGNORECASE
        )

        if match:
            ticket_id = int(match.group(1))
            label = match.group(2).capitalize()

            if label.lower() in valid_labels:
                predictions[ticket_id] = label

    return predictions


v1_predictions = parse_batch_predictions(batch_v1_output)
v2_predictions = parse_batch_predictions(batch_v2_output)
v3_predictions = parse_batch_predictions(batch_v3_output)

print("V1 parsed:", len(v1_predictions))
print("V2 parsed:", len(v2_predictions))
print("V3 parsed:", len(v3_predictions))

V1 parsed: 12
V2 parsed: 12
V3 parsed: 12


In [19]:
classification_eval = evaluation_df[
    ["eval_id", "subject", "type"]
].copy()

classification_eval["v1_prediction"] = (
    classification_eval["eval_id"].map(v1_predictions)
)

classification_eval["v2_prediction"] = (
    classification_eval["eval_id"].map(v2_predictions)
)

classification_eval["v3_prediction"] = (
    classification_eval["eval_id"].map(v3_predictions)
)

classification_eval

,eval_id,subject,type,v1_prediction,v2_prediction,v3_prediction
0,1,No subject,Change,Change,Change,Change
1,2,Support Request for Billing Details Update,Change,Request,Request,Request
2,3,Enhancements to User Interface for Project Man...,Change,Change,Change,Change
3,4,Problem with SaaS Dashboard Display,Incident,Incident,Problem,Problem
4,5,Urgent Assistance Required for Medical Data Se...,Incident,Incident,Incident,Incident
5,6,No subject,Incident,Incident,Incident,Incident
6,7,Challenges in Marketing Campaigns,Problem,Problem,Problem,Problem
7,8,Problem Regarding Data Encryption,Problem,Incident,Problem,Problem
8,9,Network Connectivity Problem,Problem,Incident,Problem,Incident
9,10,Support for Integrating Drupal with Magento,Request,Request,Request,Request


In [20]:
classification_eval["v1_correct"] = (
    classification_eval["type"]
    == classification_eval["v1_prediction"]
)

classification_eval["v2_correct"] = (
    classification_eval["type"]
    == classification_eval["v2_prediction"]
)

classification_eval["v3_correct"] = (
    classification_eval["type"]
    == classification_eval["v3_prediction"]
)


v1_accuracy = classification_eval["v1_correct"].mean()
v2_accuracy = classification_eval["v2_correct"].mean()
v3_accuracy = classification_eval["v3_correct"].mean()

print(f"V1 Accuracy: {v1_accuracy:.2%}")
print(f"V2 Accuracy: {v2_accuracy:.2%}")
print(f"V3 Accuracy: {v3_accuracy:.2%}")

V1 Accuracy: 75.00%
V2 Accuracy: 83.33%
V3 Accuracy: 75.00%


In [21]:
total = len(classification_eval)

v1_format = classification_eval["v1_prediction"].notna().sum() / total
v2_format = classification_eval["v2_prediction"].notna().sum() / total
v3_format = classification_eval["v3_prediction"].notna().sum() / total

print(f"V1 Format Adherence: {v1_format:.2%}")
print(f"V2 Format Adherence: {v2_format:.2%}")
print(f"V3 Format Adherence: {v3_format:.2%}")

V1 Format Adherence: 100.00%
V2 Format Adherence: 100.00%
V3 Format Adherence: 100.00%


In [22]:
classification_eval.to_csv(
    "../outputs/classification_evaluation.csv",
    index=False
)

print("Classification evaluation saved.")

Classification evaluation saved.


In [25]:
classification_summary = pd.DataFrame({
    "prompt_version": ["V1", "V2", "V3"],
    "accuracy": [
        v1_accuracy,
        v2_accuracy,
        v3_accuracy
    ],
    "format_adherence": [
        v1_format,
        v2_format,
        v3_format
    ]
})

classification_summary

,prompt_version,accuracy,format_adherence
0,V1,0.750000,1.0
1,V2,0.833333,1.0
2,V3,0.750000,1.0


In [26]:
classification_summary.to_csv(
    "../outputs/classification_summary.csv",
    index=False
)

print("Classification summary saved.")

Classification summary saved.
